# 🤝 使用 ADK 实现 Agent 间通信（A2A）

**欢迎来到 Kaggle 5 天 Agents 课程！**

本 Notebook 将教你构建可协作的**多智能体系统**：不同 Agent 通过 **Agent2Agent (A2A) Protocol** 通信协作。你将学习如何对接外部 Agent 服务，并像本地 Agent 一样消费远程 Agent。

在本 Notebook 中，你将：

- 理解 A2A 协议，以及它与 sub-agents 的使用边界
- 学习常见 A2A 架构模式（跨框架、跨语言、跨组织）
- 使用 `to_a2a()` 将 ADK Agent 暴露为 A2A 服务
- 使用 `RemoteA2aAgent` 消费远程 Agent
- 构建一个产品目录集成系统

## 📚 Agent2Agent (A2A) 概览

### 🤔 问题

随着系统复杂度上升，你会遇到：
- **单个 Agent 无法包办所有能力**：按领域拆分专家 Agent 更有效
- **Agent 之间需要协作**：客服要查商品，订单系统要查库存
- **不同团队会构建不同 Agent**：你希望集成外部供应商 Agent
- **语言与框架可能不同**：需要统一通信协议

### ✅ 解决方案：A2A Protocol

[Agent2Agent (A2A) Protocol](https://a2a-protocol.org/) 是一个**标准协议**，允许 Agent：
- ✨ **跨网络通信** - Agent 可部署在不同机器
- ✨ **互用能力** - 一个 Agent 可像调用工具一样调用另一个 Agent
- ✨ **跨框架协作** - 与语言/框架无关
- ✨ **遵循正式契约** - 通过 agent card 描述能力

### 🏗️ 常见 A2A 架构模式

A2A 在以下三类场景尤其有价值：

![When to choose A2A?](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day5/a2a_01.png)


1. **跨框架集成**：ADK Agent 与其他 Agent 框架协作
2. **跨语言通信**：Python Agent 调用 Java/Node.js Agent  
3. **跨组织边界**：你方内部 Agent 集成外部供应商服务

### 📋 本 Notebook 演示内容

我们将构建一个电商集成示例：
1. **Product Catalog Agent**（通过 A2A 暴露）- 外部供应商服务，提供商品信息
2. **Customer Support Agent**（消费方）- 你方客服 Agent，通过查询商品数据回答用户

```text
┌──────────────────────┐           ┌──────────────────────┐
│ Customer Support     │  ─A2A──▶  │ Product Catalog      │
│ Agent (Consumer)     │           │ Agent (Vendor)       │
│ Your Company         │           │ External Service     │
│ (localhost:8000)     │           │ (localhost:8001)     │
└──────────────────────┘           └──────────────────────┘
```

**为什么这里适合 A2A：**
- Product Catalog 由外部供应商维护（你无法改其代码）
- 两个组织、两套系统
- 需要明确正式契约
- Product Catalog 甚至可能使用不同语言/框架

### 💡 A2A vs 本地 Sub-Agents：决策表

| 因素 | 使用 A2A | 使用本地 Sub-Agents |
|--------|---------|---------------------|
| **Agent 位置** | 外部服务、不同代码库 | 同一代码库、内部调用 |
| **所有权** | 不同团队/组织 | 你的团队 |
| **网络形态** | 部署在不同机器 | 同进程/同机器 |
| **性能诉求** | 可接受网络延迟 | 要求低延迟 |
| **语言/框架** | 需要跨语言/跨框架 | 同语言 |
| **契约要求** | 需要正式 API 契约 | 内部接口即可 |
| **示例** | 外部供应商商品目录 | 内部订单处理步骤 |

### 📝 教程上下文

**本教程中**，我们会在本地模拟两类 Agent（都运行在 localhost）用于学习。实际生产中：
- Product Catalog Agent 会运行在供应商基础设施（如 `https://vendor.com`）
- Customer Support Agent 会运行在你方基础设施

## ⚙️ 环境准备

在进入今天内容前，请按下列步骤完成环境设置。

### 安装依赖

Kaggle Notebooks 环境已预装 Python 版 [google-adk](https://google.github.io/adk-docs/) 及其依赖，因此本 Notebook 无需额外安装。

若你要在课程外的本地 Python 环境安装 ADK（含 A2A 依赖），可运行：

```
pip install -q google-adk[a2a]
```

### 配置 Gemini API Key

本 Notebook 使用 [Gemini API](https://ai.google.dev/gemini-api/)，需要 API key。

**1. 获取 API key**

如果你还没有，可在 [Google AI Studio 创建 API key](https://aistudio.google.com/app/api-keys)。

**2. 将 key 添加到 Kaggle Secrets**

接下来，把 API key 作为 Kaggle User Secret 添加到 Notebook。

1. 在 Notebook 编辑器顶部菜单选择 `Add-ons` -> `Secrets`。
2. 新建标签为 `GOOGLE_API_KEY` 的 secret。
3. 把 API key 粘贴到 "Value" 字段并点击 "Save"。
4. 确认 `GOOGLE_API_KEY` 左侧复选框已勾选，使其附加到该 Notebook。

**3. 在 Notebook 中鉴权**

运行下面代码单元，读取你保存的 `GOOGLE_API_KEY` 并设置为环境变量：

In [2]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Setup and authentication complete.


### 导入 ADK 组件

现在导入你将用到的 Agent Development Kit 与 Generative AI 组件，保证代码结构清晰并具备必要构建块。

In [3]:
import json
import requests
import subprocess
import time
import uuid

from google.adk.agents import LlmAgent
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
    AGENT_CARD_WELL_KNOWN_PATH,
)

from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Hide additional warnings in the notebook
import warnings

warnings.filterwarnings("ignore")

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### 配置重试选项

在使用 LLM 时，你可能会遇到速率限制或服务临时不可用等瞬时错误。重试选项会通过指数退避自动重试请求。

In [4]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

## 📦 第 1 部分：创建将被暴露的 Product Catalog Agent

我们将构建一个**Product Catalog Agent**，用于从外部供应商目录返回商品信息。该 Agent 会**通过 A2A 暴露**，供其他 Agent（如客服）调用。

### 为什么要暴露这个 Agent？
- 在真实系统中，它通常由**外部供应商**维护
- 你的内部 Agent（客服、销售、库存）都需要商品数据
- 供应商**掌控自己的代码库**，你无法直接修改实现
- 通过 A2A 暴露后，授权 Agent 都能按标准协议调用

In [5]:
# Define a product catalog lookup tool
# In a real system, this would query the vendor's product database
def get_product_info(product_name: str) -> str:
    """Get product information for a given product.

    Args:
        product_name: Name of the product (e.g., "iPhone 15 Pro", "MacBook Pro")

    Returns:
        Product information as a string
    """
    # Mock product catalog - in production, this would query a real database
    product_catalog = {
        "iphone 15 pro": "iPhone 15 Pro, $999, Low Stock (8 units), 128GB, Titanium finish",
        "samsung galaxy s24": "Samsung Galaxy S24, $799, In Stock (31 units), 256GB, Phantom Black",
        "dell xps 15": 'Dell XPS 15, $1,299, In Stock (45 units), 15.6" display, 16GB RAM, 512GB SSD',
        "macbook pro 14": 'MacBook Pro 14", $1,999, In Stock (22 units), M3 Pro chip, 18GB RAM, 512GB SSD',
        "sony wh-1000xm5": "Sony WH-1000XM5 Headphones, $399, In Stock (67 units), Noise-canceling, 30hr battery",
        "ipad air": 'iPad Air, $599, In Stock (28 units), 10.9" display, 64GB',
        "lg ultrawide 34": 'LG UltraWide 34" Monitor, $499, Out of Stock, Expected: Next week',
    }

    product_lower = product_name.lower().strip()

    if product_lower in product_catalog:
        return f"Product: {product_catalog[product_lower]}"
    else:
        available = ", ".join([p.title() for p in product_catalog.keys()])
        return f"Sorry, I don't have information for {product_name}. Available products: {available}"


# Create the Product Catalog Agent
# This agent specializes in providing product information from the vendor's catalog
product_catalog_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="product_catalog_agent",
    description="External vendor's product catalog agent that provides product information and availability.",
    instruction="""
    You are a product catalog specialist from an external vendor.
    When asked about products, use the get_product_info tool to fetch data from the catalog.
    Provide clear, accurate product information including price, availability, and specs.
    If asked about multiple products, look up each one.
    Be professional and helpful.
    """,
    tools=[get_product_info],  # Register the product lookup tool
)

print("✅ Product Catalog Agent created successfully!")
print("   Model: gemini-2.5-flash-lite")
print("   Tool: get_product_info()")
print("   Ready to be exposed via A2A...")

✅ Product Catalog Agent created successfully!
   Model: gemini-2.5-flash-lite
   Tool: get_product_info()
   Ready to be exposed via A2A...


## 🌐 第 2 部分：通过 A2A 暴露 Product Catalog Agent

现在我们使用 ADK 的 `to_a2a()`，让 Product Catalog Agent 可被其他 Agent 访问。

### `to_a2a()` 做了什么：
- 🔧 把你的 Agent 包装为 A2A 兼容服务（FastAPI/Starlette）
- 📋 自动生成 **agent card**，包含：
  - Agent 名称、描述、版本
  - Skills（你的 tools/functions 在 A2A 中会映射为 skills）
  - 协议版本与端点
  - 输入/输出模式
- 🌐 在 `/.well-known/agent-card.json` 提供 agent card（A2A 标准路径）
- ✨ 自动处理 A2A 协议细节（请求/响应格式、task 端点等）

这是把 ADK Agent 暴露为 A2A 的**最简方式**。

**💡 核心概念：Agent Card**

**agent card** 是一个 JSON 文档，相当于 Agent 的“名片”，用于描述：
- Agent 是做什么的（name、description、version）
- 有哪些能力（skills、tools、functions）  
- 如何通信（URL、protocol version、endpoints）

每个 A2A Agent 都必须在标准路径发布 agent card：`/.well-known/agent-card.json`

它就是告诉其他 Agent “如何与你协作”的正式契约。

📖 **更多参考：**
- [Exposing Agents with ADK](https://google.github.io/adk-docs/a2a/quickstart-exposing/)
- [A2A Protocol Specification](https://a2a-protocol.org/latest/specification/)

In [6]:
# Convert the product catalog agent to an A2A-compatible application
# This creates a FastAPI/Starlette app that:
#   1. Serves the agent at the A2A protocol endpoints
#   2. Provides an auto-generated agent card
#   3. Handles A2A communication protocol
product_catalog_a2a_app = to_a2a(
    product_catalog_agent, port=8001  # Port where this agent will be served
)

print("✅ Product Catalog Agent is now A2A-compatible!")
print("   Agent will be served at: http://localhost:8001")
print("   Agent card will be at: http://localhost:8001/.well-known/agent-card.json")
print("   Ready to start the server...")

✅ Product Catalog Agent is now A2A-compatible!
   Agent will be served at: http://localhost:8001
   Agent card will be at: http://localhost:8001/.well-known/agent-card.json
   Ready to start the server...


## 🚀 第 3 部分：启动 Product Catalog Agent 服务

我们将用 `uvicorn` 在**后台**启动 Product Catalog Agent 服务，使其可被其他 Agent 调用。

### 为什么要后台运行？
- 在创建并测试 Customer Support Agent 时，服务需持续可用
- 这模拟了真实场景：不同 Agent 作为独立服务运行
- 生产环境中通常由供应商在其基础设施上托管该服务

In [7]:
# First, let's save the product catalog agent to a file that uvicorn can import
product_catalog_agent_code = '''
import os
from google.adk.agents import LlmAgent
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.models.google_llm import Gemini
from google.genai import types

retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

def get_product_info(product_name: str) -> str:
    """Get product information for a given product."""
    product_catalog = {
        "iphone 15 pro": "iPhone 15 Pro, $999, Low Stock (8 units), 128GB, Titanium finish",
        "samsung galaxy s24": "Samsung Galaxy S24, $799, In Stock (31 units), 256GB, Phantom Black",
        "dell xps 15": "Dell XPS 15, $1,299, In Stock (45 units), 15.6\\" display, 16GB RAM, 512GB SSD",
        "macbook pro 14": "MacBook Pro 14\\", $1,999, In Stock (22 units), M3 Pro chip, 18GB RAM, 512GB SSD",
        "sony wh-1000xm5": "Sony WH-1000XM5 Headphones, $399, In Stock (67 units), Noise-canceling, 30hr battery",
        "ipad air": "iPad Air, $599, In Stock (28 units), 10.9\\" display, 64GB",
        "lg ultrawide 34": "LG UltraWide 34\\" Monitor, $499, Out of Stock, Expected: Next week",
    }
    
    product_lower = product_name.lower().strip()
    
    if product_lower in product_catalog:
        return f"Product: {product_catalog[product_lower]}"
    else:
        available = ", ".join([p.title() for p in product_catalog.keys()])
        return f"Sorry, I don't have information for {product_name}. Available products: {available}"

product_catalog_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="product_catalog_agent",
    description="External vendor's product catalog agent that provides product information and availability.",
    instruction="""
    You are a product catalog specialist from an external vendor.
    When asked about products, use the get_product_info tool to fetch data from the catalog.
    Provide clear, accurate product information including price, availability, and specs.
    If asked about multiple products, look up each one.
    Be professional and helpful.
    """,
    tools=[get_product_info]
)

# Create the A2A app
app = to_a2a(product_catalog_agent, port=8001)
'''

# Write the product catalog agent to a temporary file
with open("/tmp/product_catalog_server.py", "w") as f:
    f.write(product_catalog_agent_code)

print("📝 Product Catalog agent code saved to /tmp/product_catalog_server.py")

# Start uvicorn server in background
# Note: We redirect output to avoid cluttering the notebook
server_process = subprocess.Popen(
    [
        "uvicorn",
        "product_catalog_server:app",  # Module:app format
        "--host",
        "localhost",
        "--port",
        "8001",
    ],
    cwd="/tmp",  # Run from /tmp where the file is
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env={**os.environ},  # Pass environment variables (including GOOGLE_API_KEY)
)

print("🚀 Starting Product Catalog Agent server...")
print("   Waiting for server to be ready...")

# Wait for server to start (poll until it responds)
max_attempts = 30
for attempt in range(max_attempts):
    try:
        response = requests.get(
            "http://localhost:8001/.well-known/agent-card.json", timeout=1
        )
        if response.status_code == 200:
            print(f"\n✅ Product Catalog Agent server is running!")
            print(f"   Server URL: http://localhost:8001")
            print(f"   Agent card: http://localhost:8001/.well-known/agent-card.json")
            break
    except requests.exceptions.RequestException:
        time.sleep(5)
        print(".", end="", flush=True)
else:
    print("\n⚠️  Server may not be ready yet. Check manually if needed.")

# Store the process so we can stop it later
globals()["product_catalog_server_process"] = server_process

📝 Product Catalog agent code saved to /tmp/product_catalog_server.py
🚀 Starting Product Catalog Agent server...
   Waiting for server to be ready...
.....
✅ Product Catalog Agent server is running!
   Server URL: http://localhost:8001
   Agent card: http://localhost:8001/.well-known/agent-card.json


### 🔍 查看自动生成的 Agent Card

`to_a2a()` 已自动生成描述 Product Catalog Agent 能力的 **agent card**。下面直接查看内容。

In [8]:
# Fetch the agent card from the running server
try:
    response = requests.get(
        "http://localhost:8001/.well-known/agent-card.json", timeout=5
    )

    if response.status_code == 200:
        agent_card = response.json()
        print("📋 Product Catalog Agent Card:")
        print(json.dumps(agent_card, indent=2))

        print("\n✨ Key Information:")
        print(f"   Name: {agent_card.get('name')}")
        print(f"   Description: {agent_card.get('description')}")
        print(f"   URL: {agent_card.get('url')}")
        print(f"   Skills: {len(agent_card.get('skills', []))} capabilities exposed")
    else:
        print(f"❌ Failed to fetch agent card: {response.status_code}")

except requests.exceptions.RequestException as e:
    print(f"❌ Error fetching agent card: {e}")
    print("   Make sure the Product Catalog Agent server is running (previous cell)")

📋 Product Catalog Agent Card:
{
  "capabilities": {},
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "description": "External vendor's product catalog agent that provides product information and availability.",
  "name": "product_catalog_agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "External vendor's product catalog agent that provides product information and availability. \n    I am a product catalog specialist from an external vendor.\n    When asked about products, use the get_product_info tool to fetch data from the catalog.\n    Provide clear, accurate product information including price, availability, and specs.\n    If asked about multiple products, look up each one.\n    Be professional and helpful.\n    ",
      "id": "product_catalog_agent",
      "name": "model",
      "tags": [
        "llm"
      ]
    },
    {
      "description": "Get product information

## 🎧 第 4 部分：创建 Customer Support Agent（消费方）

现在创建**Customer Support Agent**，通过 A2A 消费 Product Catalog Agent。

### 工作方式：
1. 使用 `RemoteA2aAgent` 为 Product Catalog Agent 创建**客户端代理**
2. Customer Support Agent 可像使用普通工具一样调用 Product Catalog Agent
3. A2A 协议通信细节由 ADK 在后台自动处理

这体现了 A2A 的核心价值：**远程 Agent 的协作体验接近本地调用。**

**RemoteA2aAgent 的机制：**
- 它是读取远程 agent card 的**客户端代理**
- 会把 sub-agent 调用翻译为 A2A 协议请求（HTTP POST 到 `/tasks`）
- 封装协议细节，让你像普通 sub-agent 一样使用

📖 **更多参考：**
- [Consuming Remote Agents with ADK](https://google.github.io/adk-docs/a2a/quickstart-consuming/)
- [What is A2A?](https://a2a-protocol.org/latest/topics/what-is-a2a/)

In [9]:
# Create a RemoteA2aAgent that connects to our Product Catalog Agent
# This acts as a client-side proxy - the Customer Support Agent can use it like a local agent
remote_product_catalog_agent = RemoteA2aAgent(
    name="product_catalog_agent",
    description="Remote product catalog agent from external vendor that provides product information.",
    # Point to the agent card URL - this is where the A2A protocol metadata lives
    agent_card=f"http://localhost:8001{AGENT_CARD_WELL_KNOWN_PATH}",
)

print("✅ Remote Product Catalog Agent proxy created!")
print(f"   Connected to: http://localhost:8001")
print(f"   Agent card: http://localhost:8001{AGENT_CARD_WELL_KNOWN_PATH}")
print("   The Customer Support Agent can now use this like a local sub-agent!")

✅ Remote Product Catalog Agent proxy created!
   Connected to: http://localhost:8001
   Agent card: http://localhost:8001/.well-known/agent-card.json
   The Customer Support Agent can now use this like a local sub-agent!


In [10]:
# Now create the Customer Support Agent that uses the remote Product Catalog Agent
customer_support_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="customer_support_agent",
    description="A customer support assistant that helps customers with product inquiries and information.",
    instruction="""
    You are a friendly and professional customer support agent.
    
    When customers ask about products:
    1. Use the product_catalog_agent sub-agent to look up product information
    2. Provide clear answers about pricing, availability, and specifications
    3. If a product is out of stock, mention the expected availability
    4. Be helpful and professional!
    
    Always get product information from the product_catalog_agent before answering customer questions.
    """,
    sub_agents=[remote_product_catalog_agent],  # Add the remote agent as a sub-agent!
)

print("✅ Customer Support Agent created!")
print("   Model: gemini-2.5-flash-lite")
print("   Sub-agents: 1 (remote Product Catalog Agent via A2A)")
print("   Ready to help customers!")

✅ Customer Support Agent created!
   Model: gemini-2.5-flash-lite
   Sub-agents: 1 (remote Product Catalog Agent via A2A)
   Ready to help customers!


## 🧪 第 5 部分：测试 A2A 通信

下面测试 agent-to-agent 通信。我们向 Customer Support Agent 提问商品问题，它会通过 A2A 与 Product Catalog Agent 通信。

### 幕后发生了什么：
1. 客户向 Support Agent 询问某商品
2. Support Agent 判断需要商品信息
3. Support Agent 调用 `remote_product_catalog_agent`（RemoteA2aAgent）
4. ADK 向 `http://localhost:8001` 发起 A2A 协议请求
5. Product Catalog Agent 处理请求并返回
6. Support Agent 接收结果后继续推理
7. 客户得到最终回答

整个过程对上层几乎**透明**：Support Agent 无需感知对方是远程 Agent。

In [11]:
async def test_a2a_communication(user_query: str):
    """
    Test the A2A communication between Customer Support Agent and Product Catalog Agent.

    This function:
    1. Creates a new session for this conversation
    2. Sends the query to the Customer Support Agent
    3. Support Agent communicates with Product Catalog Agent via A2A
    4. Displays the response

    Args:
        user_query: The question to ask the Customer Support Agent
    """
    # Setup session management (required by ADK)
    session_service = InMemorySessionService()

    # Session identifiers
    app_name = "support_app"
    user_id = "demo_user"
    # Use unique session ID for each test to avoid conflicts
    session_id = f"demo_session_{uuid.uuid4().hex[:8]}"

    # CRITICAL: Create session BEFORE running agent (synchronous, not async!)
    # This pattern matches the deployment notebook exactly
    session = await session_service.create_session(
        app_name=app_name, user_id=user_id, session_id=session_id
    )

    # Create runner for the Customer Support Agent
    # The runner manages the agent execution and session state
    runner = Runner(
        agent=customer_support_agent, app_name=app_name, session_service=session_service
    )

    # Create the user message
    # This follows the same pattern as the deployment notebook
    test_content = types.Content(parts=[types.Part(text=user_query)])

    # Display query
    print(f"\n👤 Customer: {user_query}")
    print(f"\n🎧 Support Agent response:")
    print("-" * 60)

    # Run the agent asynchronously (handles streaming responses and A2A communication)
    async for event in runner.run_async(
        user_id=user_id, session_id=session_id, new_message=test_content
    ):
        # Print final response only (skip intermediate events)
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if hasattr(part, "text"):
                    print(part.text)

    print("-" * 60)


# Run the test
print("🧪 Testing A2A Communication...\n")
await test_a2a_communication("Can you tell me about the iPhone 15 Pro? Is it in stock?")

🧪 Testing A2A Communication...


👤 Customer: Can you tell me about the iPhone 15 Pro? Is it in stock?

🎧 Support Agent response:
------------------------------------------------------------


INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash-lite, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.agents.remote_a2a_agent:Successfully resolved remote A2A agent: product_catalog_agent


The iPhone 15 Pro is currently in stock, with only 8 units available. It costs $999 and comes with a 128GB storage capacity and a titanium finish.
------------------------------------------------------------


### 试试更多示例

再测试几个场景，观察 A2A 通信效果。

In [12]:
# Test comparing multiple products
await test_a2a_communication(
    "I'm looking for a laptop. Can you compare the Dell XPS 15 and MacBook Pro 14 for me?"
)

INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash-lite, backend: GoogleLLMVariant.GEMINI_API, stream: False



👤 Customer: I'm looking for a laptop. Can you compare the Dell XPS 15 and MacBook Pro 14 for me?

🎧 Support Agent response:
------------------------------------------------------------


INFO:google_adk.google.adk.models.google_llm:Response received from the model.


I can help you with that! Here's a comparison of the Dell XPS 15 and the MacBook Pro 14:

**Dell XPS 15:**
*   **Price:** $1,299
*   **Availability:** In Stock (45 units)
*   **Specs:** 15.6" display, 16GB RAM, 512GB SSD

**MacBook Pro 14":**
*   **Price:** $1,999
*   **Availability:** In Stock (22 units)
*   **Specs:** M3 Pro chip, 18GB RAM, 512GB SSD

Do you have any other questions about these laptops or would you like to explore other options?
------------------------------------------------------------


In [13]:
# Test specific product inquiry
await test_a2a_communication(
    "Do you have the Sony WH-1000XM5 headphones? What's the price?"
)

INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash-lite, backend: GoogleLLMVariant.GEMINI_API, stream: False



👤 Customer: Do you have the Sony WH-1000XM5 headphones? What's the price?

🎧 Support Agent response:
------------------------------------------------------------


INFO:google_adk.google.adk.models.google_llm:Response received from the model.


The Sony WH-1000XM5 headphones are available for $399 and are currently in stock with 67 units available. They feature noise-canceling technology and a 30-hour battery life.
------------------------------------------------------------


## 🔍 第 6 部分：理解刚才发生了什么

### A2A 通信流程

当你运行上面的测试时，Agent 间通信的详细流程如下：

![](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day5/a2a_03.png)

**A2A 协议层通信：**

在协议层面，核心过程是：
- **RemoteA2aAgent** 向 `http://localhost:8001` 的 `/tasks` 端点发送 HTTP POST
- 请求与响应遵循 [A2A Protocol Specification](https://a2a-protocol.org/latest/specification/)
- 数据以标准化 JSON 交换
- 协议保证任意 A2A 兼容 Agent（不受语言/框架限制）可互通

这种标准化正是跨组织、跨语言 Agent 协作成为可能的关键。

---

**本次调用发生了什么：**
1. **客户**询问 iPhone 15 Pro
2. **Customer Support Agent**（LlmAgent）收到问题并判断需要商品信息
3. **Support Agent** 将任务委派给 `product_catalog_agent` 子 Agent
4. **RemoteA2aAgent**（客户端代理）把调用转换为 A2A 请求
5. A2A 请求通过 HTTP 发送到 `http://localhost:8001`（图中黄色高亮）
6. **Product Catalog Agent**（服务端）处理请求并调用 `get_product_info("iPhone 15 Pro")`
7. **Product Catalog Agent** 通过 A2A 响应返回商品信息
8. **RemoteA2aAgent** 接收响应并回传给 Support Agent
9. **Support Agent** 组织最终答案
10. **客户**收到完整有效回复

### 关键收益

1. **透明性**：Support Agent 不需要显式“知道”对方是远程 Agent
2. **标准协议**：遵循 A2A 标准，任意兼容 Agent 都能接入
3. **易集成**：只需一行 `sub_agents=[remote_product_catalog_agent]`
4. **关注点分离**：商品逻辑在 Catalog Agent（供应商侧），客服逻辑在 Support Agent（你方）

### 真实场景应用

该模式可支持：
- **微服务化**：每个 Agent 都是独立服务
- **第三方集成**：接入外部供应商 Agent（商品目录、支付处理等）
- **跨语言协作**：Catalog Agent 可用 Java，Support Agent 可用 Python
- **团队专业分工**：供应商维护目录，你方维护客服 Agent
- **跨组织协作**：供应商托管其服务，你方通过 A2A 集成

## 🎓 下一步与学习资源

### 🚀 可扩展方向

掌握 A2A 基础后，你可以尝试：

1. **增加更多 Agent**：
   - 新建 **Inventory Agent**（库存与补货计划）
   - 新建 **Shipping Agent**（配送时效与追踪）
   - 让 Customer Support Agent 通过 A2A 协调三者

2. **接入真实数据源**：
   - 用真实数据库（PostgreSQL、MongoDB）替换 mock 商品目录
   - 接入真实库存系统
   - 对接真实支付网关 API

3. **进阶 A2A 能力**：
   - 在 Agent 间实现鉴权（API key、OAuth）
   - 增加网络失败下的错误处理与重试
   - 尝试替代方案 `adk api_server --a2a`

4. **部署到生产环境**：
   - 把 Product Catalog Agent 部署到 Agent Engine
   - 将 agent card URL 指向生产地址（例如 `https://vendor-catalog.example.com`）
   - 消费方 Agent 即可通过互联网接入

### 📚 文档

**A2A Protocol：**

- [Official A2A Protocol Website](https://a2a-protocol.org/)
- [A2A Protocol Specification](https://a2a-protocol.org/latest/spec/)

**ADK A2A 指南：**

- [Introduction to A2A in ADK](https://google.github.io/adk-docs/a2a/intro/)
- [Exposing Agents Quickstart](https://google.github.io/adk-docs/a2a/quickstart-exposing/)
- [Consuming Agents Quickstart](https://google.github.io/adk-docs/a2a/quickstart-consuming/)

**其他部署选项：**

- [Deploy ADK Agents to Cloud Run](https://google.github.io/adk-docs/deploy/cloud-run/)
- [Deploy to Agent Engine](https://google.github.io/adk-docs/deploy/agent-engine/)
- [Deploy to GKE](https://google.github.io/adk-docs/deploy/gke/)

---

## 📊 总结 - A2A 通信模式

### 关键收获

在本 Notebook 中，你学习了如何使用 A2A 构建多智能体系统：

- **A2A Protocol**：跨网络、跨框架的标准 Agent 通信协议
- **暴露 Agent**：使用 `to_a2a()` 把 Agent 暴露为可被访问的服务，并自动生成 agent card
- **消费 Agent**：使用 `RemoteA2aAgent` 将远程 Agent 当作本地 sub-agent 集成
- **适用场景**：微服务架构、跨团队集成、第三方 Agent 能力接入

---

## ✅ 恭喜！你已经掌握 A2A

你已经成功学会如何用 A2A 协议构建多智能体系统。

现在你知道如何把 Agent 暴露为服务、如何消费远程 Agent，以及如何构建可跨团队与组织扩展的协作式多 Agent 系统。

**ℹ️ 说明：无需提交！**

这个 Notebook 仅用于动手实践与学习。完成课程**不需要**把它提交到任何地方。

### 📚 继续学习

可参考以下文档：

- [ADK Documentation](https://google.github.io/adk-docs/)
- [A2A Protocol Official Website](https://a2a-protocol.org/)
- [A2A Tutorials](https://a2a-protocol.org/latest/tutorials/)
- [Introduction to A2A in ADK](https://google.github.io/adk-docs/a2a/intro/)
- [Exposing Agents Quickstart](https://google.github.io/adk-docs/a2a/quickstart-exposing/)
- [Consuming Agents Quickstart](https://google.github.io/adk-docs/a2a/quickstart-consuming/)

### 🎯 下一步

掌握 A2A 后，你已经可以构建“多专家协作”的复杂系统来解决真实问题。你也可以把 Agent 部署到 Cloud Run 或 Agent Engine，让它们通过互联网可访问。

准备继续进阶？建议探索 ADK 的高级能力：custom agents、streaming、生产级部署模式。